# Dia 20 — O script quebrado 🧰

**Linux para OSINT — 21 dias** · baseado no curso de [cipher387](https://github.com/cipher387/linux-for-OSINT-21-day) (MIT)

📘 Texto completo do dia: [dia-20-ferramentas-uteis](https://github.com/Ridd1kulusC0d3r/linux-para-osint-21-dias/tree/main/dias/dia-20-ferramentas-uteis)

## 🎬 Missão

> Na apreensão do notebook de um afiliado, a perícia achou dois arquivos
> d'Os Vazantes: um script chamado `selo.sh` e uma cópia do histórico do
> terminal de maré_alta. O histórico termina assim:
>
> ```text
> bash selo.sh
> bash selo.sh
> shellcheck selo.sh
> history -c
> ```
>
> — Rodou duas vezes, não funcionou, chamou o shellcheck e apagou o histórico
> — lê Bento. — Ele nunca conseguiu consertar.
>
> A Mestra Iara empurra a cadeira para você.
> — Recruta, vinte dias atrás você não sabia o que era `ls`. Hoje conserta o
> script do chefe da quadrilha. **Sem mexer no original.**

**Sua tarefa:** usar o `shellcheck` para achar os erros de
`desafio/script_quebrado.sh`, criar uma cópia corrigida, rodá-la e obter o
selo — a flag `VAZANTE{...}`.

## ⚙️ Preparação

Rode a célula abaixo **uma vez**. Ela baixa o repositório para dentro do Colab e define a variável `L4O_HOME`, que todas as outras células usam.

> No Colab você é `root` e o `apt` funciona. Cada célula `%%bash` roda num processo separado: variáveis e `cd` não passam de uma célula para outra.

In [ ]:
import os, subprocess
from pathlib import Path

# Se já estamos dentro do repositório (Codespaces, máquina local), usa ele.
aqui = Path.cwd()
candidatos = [aqui, *aqui.parents]
raiz = next((p for p in candidatos if (p / 'lib' / 'comum.sh').exists()), None)

if raiz is None:  # Colab: clona o repositório
    raiz = Path('/content/linux-para-osint-21-dias')
    if not raiz.exists():
        subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Ridd1kulusC0d3r/linux-para-osint-21-dias', str(raiz)], check=True)

os.environ['L4O_HOME'] = str(raiz)
os.environ['L4O_CORES'] = '1'
print('Repositório em', raiz)

### shellcheck: o corretor ortográfico do Bash

```text
O programa shellcheck lê um script SEM executá-lo e aponta erros e armadilhas.
Cada aviso tem um código (SC2086, SC2283...) com explicação na wiki:
  https://www.shellcheck.net/wiki/SC2086
Níveis: error (vai quebrar), warning (provavelmente errado),
        info/style (melhor prática).
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-20-ferramentas-uteis" && source "$L4O_HOME/lib/comum.sh"
titulo "shellcheck no script de maré_alta"
precisa shellcheck
shellcheck desafio/script_quebrado.sh || true      # || true: avisos não param o curso
echo "--- formato resumido (uma linha por aviso) ---"
shellcheck -f gcc desafio/script_quebrado.sh || true

### O exemplo clássico do cipher387

```text
O script original do curso (wrong_code.sh) tem erros "de lógica" que só o
o shellcheck vê: uma opção -v tratada no case mas não declarada no getopts,
e $$(uname), que vira o número do processo + "(uname)".
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-20-ferramentas-uteis" && source "$L4O_HOME/lib/comum.sh"
titulo "Erros que o Bash não reclama, mas o shellcheck sim"
precisa shellcheck
mkdir -p treino
cat > treino/wrong_code.sh <<'FIM'
#!/bin/bash
while getopts "nf:" param; do
  case "$param" in
    f) file="$OPTARG" ;;
    v) set -x ;;
  esac
done
case "$file" in
  *.gz) gzip -d "$file" ;;
  *) echo "Unknown filetype" ;;
esac
if [[ "$$(uname)" == "Linux" ]]; then echo "Using Linux"; fi
FIM
bash treino/wrong_code.sh            # roda "sem erro"... e não faz o que devia
shellcheck treino/wrong_code.sh || true
rm -r treino

### Rodando o script quebrado

```text
Do jeito que está, o script de maré_alta falha. Leia a mensagem de erro:
"pasta: command not found" — o Bash achou que "pasta" era um COMANDO.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-20-ferramentas-uteis" && source "$L4O_HOME/lib/comum.sh"
titulo "Executando o script quebrado"
bash desafio/script_quebrado.sh || alerta "falhou, como esperado"

### Os três erros mais comuns de Bash

```text
1) Espaço em volta do =    nome = "x"   -> roda o comando "nome" com args "=" e "x"
2) Variável sem aspas      $pasta       -> "pedacos de selo" vira 3 palavras
3) Nome de arquivo com espaço sem aspas -> "parte 01.txt" vira "parte" e "01.txt"
Regra de ouro: SEMPRE "$variavel" entre aspas duplas.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-20-ferramentas-uteis" && source "$L4O_HOME/lib/comum.sh"
titulo "Demonstração: aspas fazem diferença"
mkdir -p "treino/pasta com espaco"
echo "oi" > "treino/pasta com espaco/arquivo 1.txt"
pasta="treino/pasta com espaco"
echo "--- sem aspas (quebra nos espaços) ---"
# shellcheck disable=SC2231  # (erro proposital, para você ver o problema)
for a in $pasta/*.txt; do echo "[$a]"; done
echo "--- com aspas ---"
for a in "$pasta"/*.txt; do echo "[$a]"; done
echo "--- set -x mostra cada comando ANTES de rodar (modo depuração) ---"
# shellcheck disable=SC2016  # (aspas simples de propósito: o $p é do bash -c)
bash -x -c 'p="a b"; ls $p' 2>&1 | head -n 3 || true
rm -r treino

### history: tudo que você digitou

```text
O Bash guarda os comandos digitados em ~/.bash_history.
  history          lista com números
  history | grep   procura um comando antigo
  !123             repete o comando número 123
  !!               repete o último (sudo !! = o último, com sudo)
  Ctrl+R           busca reversa interativa: digite um pedaço e aperte Ctrl+R de novo
Num script não há o que você digitou, então carregamos (history -r) um
histórico "esquecido" por maré_alta (uma cópia, para não tocar no seu).
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-20-ferramentas-uteis" && source "$L4O_HOME/lib/comum.sh"
titulo "O histórico de maré_alta"
HISTFILE=$(mktemp)                       # cópia temporária
cp desafio/historico_mare_alta.txt "$HISTFILE"
history -c                               # começa vazio
history -r "$HISTFILE"                   # lê o arquivo para a memória
history
echo "--- só os comandos de git ---"
history | grep ' git '
echo "--- os 3 últimos ---"
history 3
rm -f "$HISTFILE"
dica "O último comando dele foi 'history -c' (apagar o histórico)... mas o arquivo já estava salvo."

### alias: apelidos para comandos longos

```text
alias cria um atalho. Para ficar permanente, coloque no ~/.bashrc.
Em scripts, aliases vêm desligados; `shopt -s expand_aliases` liga.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-20-ferramentas-uteis" && source "$L4O_HOME/lib/comum.sh"
titulo "Aliases"
shopt -s expand_aliases
alias ll='ls -lah'
alias meuip='curl -s https://ifconfig.me; echo'
alias hashes='sha256sum'
alias                                    # lista os aliases definidos
ll desafio
mostrar "echo \"alias ll='ls -lah'\" >> ~/.bashrc && source ~/.bashrc   # permanente"

### Completar com Tab e outras teclas que economizam a vida

```text
Tab        completa nomes de comandos, arquivos e pastas (2x Tab = opções)
Ctrl+R     busca no histórico            Ctrl+A / Ctrl+E  início / fim da linha
Ctrl+C     cancela o comando             Ctrl+L           limpa a tela
Ctrl+U     apaga a linha                 Alt+.            último argumento do comando anterior
Tab é a melhor amiga de quem tem pasta com espaço: ela põe as \ ou aspas sozinha.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-20-ferramentas-uteis" && source "$L4O_HOME/lib/comum.sh"
titulo "Teclas de atalho"
dica "Digite: cat desafio/ped<Tab>  -> vira  cat desafio/pedacos\\ de\\ selo/"
complete -p git 2>/dev/null || dica "No seu terminal, 'complete -p' lista as regras de autocompletar."

### tldr: o manual resumido

```text
man é completo, mas longo. tldr mostra só os exemplos mais usados.
Na primeira vez ele baixa as páginas (precisa de internet).
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-20-ferramentas-uteis" && source "$L4O_HOME/lib/comum.sh"
titulo "tldr"
precisa tldr
if online; then
  tldr tar 2>/dev/null | head -n 25 || alerta "tldr não conseguiu baixar as páginas"
else
  alerta "offline: tldr precisa baixar as páginas na primeira vez"
  mostrar "tldr tar"
  mostrar "tldr --update"
fi

### fzf: busca difusa em qualquer lista

```text
fzf recebe uma lista e deixa você filtrar digitando pedaços (fuzzy).
Interativo no terminal; com --filter funciona em script.
Depois de instalado, Ctrl+R e Ctrl+T no terminal ganham superpoderes com ele.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-20-ferramentas-uteis" && source "$L4O_HOME/lib/comum.sh"
titulo "fzf"
precisa fzf
find desafio -type f | fzf --filter='parte 03'
echo "--- 'difuso': letras na ordem, não precisam estar juntas ---"
find desafio -type f | fzf --filter='hstmar'
interativo fzf

### explainshell: o que esse comando faz?

```text
explainshell.com explica cada pedaço de um comando. Montamos a URL com
urlencode (jq @uri). Bom para comandos copiados da internet ou de um chatbot
— nunca rode o que você não entende.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-20-ferramentas-uteis" && source "$L4O_HOME/lib/comum.sh"
titulo "explainshell"
precisa jq
cmd="find . -type f -name '*.txt' -exec sha256sum {} +"
echo "https://explainshell.com/explain?cmd=$(jq -rn --arg c "$cmd" '$c | @uri')"

### Depurando o script de maré_alta com sed (sem mexer no original)

```text
Copie, corrija a cópia, rode a cópia. O original fica intacto como evidência.
Aqui corrigimos SÓ o primeiro erro, para ver que ainda não basta.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-20-ferramentas-uteis" && source "$L4O_HOME/lib/comum.sh"
titulo "Corrigindo um erro de cada vez"
precisa shellcheck
cp desafio/script_quebrado.sh desafio/teste_parcial.sh
sed -i 's/^pasta = /pasta=/' desafio/teste_parcial.sh     # tira os espaços do =
grep -n '^pasta' desafio/teste_parcial.sh
bash desafio/teste_parcial.sh || alerta "ainda quebrado: faltam as aspas"
shellcheck desafio/teste_parcial.sh || true
rm desafio/teste_parcial.sh

## 🐍 Ponte Python

O mesmo resultado só com a biblioteca padrão do Python:

In [ ]:
!cd "$L4O_HOME/dias/dia-20-ferramentas-uteis" && python3 equivalente.py

## 🧩 Desafio

Encontrou a flag? Troque o texto abaixo e rode a célula.

Dicas e passo a passo estão no README do dia.

In [ ]:
flag = 'VAZANTE{troque_aqui}'  # ✏️ cole sua resposta
!cd "$L4O_HOME/dias/dia-20-ferramentas-uteis" && bash checar.sh "{flag}"

---

📝 Anote o que aprendeu no `DIARIO.md` do repositório.

Feito por [Deivison Lourenço](https://github.com/Ridd1kulusC0d3r) · curso original de [cipher387 / Cyber Detective](https://cybdetective.com)